# 3주차 실습: 한국어 텍스트 전처리

**배포용 반완성 노트북**입니다. 위에서 아래 순서로 실행하고, `TODO`와 체크포인트를 직접 완성하세요.

## 오늘의 목표
1. NFC 유니코드 정규화가 비교·검색·중복 제거에 필요한 이유를 설명한다.
2. URL·이메일·전화번호·반복 문자를 정규표현식으로 탐지하고, 삭제와 마스킹을 구분한다.
3. 공백·정규표현식·형태소 기반 토큰화를 비교한다.
4. Kiwi와 Okt의 결과를 과업에 맞게 해석한다.
5. 불용어·품사 필터가 보존하는 정보와 잃는 정보를 판단한다.

> **제출 원칙:** TODO 실행 결과와 체크포인트 답변을 남기세요. 외부 AI·자료를 사용했다면 사용 범위와 검증 과정을 기록하세요.

> **실행 안내:** 이 노트북은 Kiwi만으로 핵심 실습을 수행할 수 있습니다. Okt는 Java 환경 문제로 설치되지 않을 수 있으므로, 사용할 수 있을 때만 비교합니다.

## 0. 준비 및 실습 데이터

이 데이터는 교육용 예시입니다. 실제 리뷰·게시글을 분석할 때에는 데이터 출처·이용 조건·개인정보 포함 여부를 확인해야 합니다.

In [1]:
# 아래 주석을 해제해 실행하세요.
!pip -q install -U kiwipiepy konlpy

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88.0/88.0 MB 7.7 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.5/11.5 MB 48.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.4/19.4 MB 71.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 438.5/438.5 kB 21.3 MB/s eta 0:00:00


In [3]:

import re # 정규표현식
import unicodedata # NFD->NFC
from collections import Counter # 빈도수 체크

import pandas as pd # 표 형태 출력
from kiwipiepy import Kiwi # 형태소 분석

kiwi = Kiwi()

# Okt는 Java 설정에 따라 초기화가 실패할 수 있습니다. 실패해도 Kiwi 실습은 계속할 수 있습니다.
try:
    from konlpy.tag import Okt
    okt = Okt()
    okt_ready = True
except Exception as error: # 실패 시 제외하고 kiwi로만 진행
    okt = None
    okt_ready = False
    print('Okt를 사용할 수 없습니다. Kiwi 중심으로 진행합니다.')
    print('사유:', type(error).__name__)

reviews = [
    (1, "정말 재미있어요!!! 배우 연기가 최고네요 😊", 1),
    (2, "스토리는 평범하지만 음악이 좋아요.", 1),
    (3, "기대했는데 너무너무 지루했어요...", 0),
    (4, "영상미는 훌륭합니다. 그런데 전개가 느려요.", 1),
    (5, "별로예요 ㅠㅠ 다시 보고 싶지 않아요.", 0),
    (6, "문의는 movie@example.com 또는 010-1234-5678로 주세요.", 1),
    (7, "새 소식: https://example.com/movie 를 확인하세요!", 1),
    (8, "가족과 함께 보기 좋은 따뜻한 작품입니다.", 1),
]
df = pd.DataFrame(reviews, columns=['review_id', 'text', 'label'])
df['label_name'] = df['label'].map({1: '긍정', 0: '부정'}) # 매핑해서 새로운 열 생성
display(df) # 깔끔한 디자인?

,review_id,text,label,label_name
0,1,정말 재미있어요!!! 배우 연기가 최고네요 😊,1,긍정
1,2,스토리는 평범하지만 음악이 좋아요.,1,긍정
2,3,기대했는데 너무너무 지루했어요...,0,부정
3,4,영상미는 훌륭합니다. 그런데 전개가 느려요.,1,긍정
4,5,별로예요 ㅠㅠ 다시 보고 싶지 않아요.,0,부정
5,6,문의는 movie@example.com 또는 010-1234-5678로 주세요.,1,긍정
6,7,새 소식: https://example.com/movie 를 확인하세요!,1,긍정
7,8,가족과 함께 보기 좋은 따뜻한 작품입니다.,1,긍정


## 1. 전처리 파이프라인 이해

전처리는 하나의 정답 함수가 아닙니다. **과업·데이터·평가 기준**에 따라 필요한 규칙이 달라집니다.

`원문 → 유니코드 정규화 → 민감정보/노이즈 탐지 → 처리 정책(보존·마스킹·삭제) 선택 → 토큰화 → 품사·불용어 필터 → 텍스트 표현`

- URL·연락처는 단순 노이즈가 아니라 개인정보 또는 참조 정보일 수 있습니다.
- 이모지·반복 문자·느낌표는 감성 신호일 수 있으므로 무조건 삭제하지 않습니다.
- 이후 주차의 빈도·TF-IDF는 여기서 만든 토큰을 입력으로 사용합니다.

### 체크포인트 1

위 파이프라인에서 **반드시 보존해야 할 수 있다고 생각하는 정보**를 하나 고르고, 어떤 분석 과업에서 유용한지 1~2문장으로 설명하세요.

## 2. 유니코드 정규화: 겉보기와 내부 표현

겉보기에는 같은 한글도 내부 유니코드 표현이 달라 비교·중복 제거에 문제가 생길 수 있습니다. 분석 전에는 보통 NFC 형태로 통일합니다.

In [4]:
composed = '가'
decomposed = unicodedata.normalize('NFD', composed) # 조합형으로 변환
print('겉보기:', composed, decomposed)
print('정규화 전 동일 여부:', composed == decomposed)

# TODO 1. composed와 decomposed를 각각 NFC로 정규화하고, 동일한지 출력하세요.
nfc_composed = unicodedata.normalize('NFC', composed)
nfc_decomposed = unicodedata.normalize('NFC', decomposed)
print('NFC 정규화 후 동일 여부:', nfc_composed == nfc_decomposed)

겉보기: 가 가
정규화 전 동일 여부: False
NFC 정규화 후 동일 여부: True


In [7]:
# TODO 2. text 열 전체를 NFC로 정규화한 text_nfc 열을 만드세요.
# 힌트: apply()함수와 lambda 식을 이용하세요
df['text_nfc'] = df['text'].apply(lambda x: unicodedata.normalize('NFC', x))

if 'text_nfc' in df.columns:
    display(df[['review_id', 'text', 'text_nfc']])
else:
    print('TODO 2를 완료한 뒤 다시 실행하세요.')

,review_id,text,text_nfc
0,1,정말 재미있어요!!! 배우 연기가 최고네요 😊,정말 재미있어요!!! 배우 연기가 최고네요 😊
1,2,스토리는 평범하지만 음악이 좋아요.,스토리는 평범하지만 음악이 좋아요.
2,3,기대했는데 너무너무 지루했어요...,기대했는데 너무너무 지루했어요...
3,4,영상미는 훌륭합니다. 그런데 전개가 느려요.,영상미는 훌륭합니다. 그런데 전개가 느려요.
4,5,별로예요 ㅠㅠ 다시 보고 싶지 않아요.,별로예요 ㅠㅠ 다시 보고 싶지 않아요.
5,6,문의는 movie@example.com 또는 010-1234-5678로 주세요.,문의는 movie@example.com 또는 010-1234-5678로 주세요.
6,7,새 소식: https://example.com/movie 를 확인하세요!,새 소식: https://example.com/movie 를 확인하세요!
7,8,가족과 함께 보기 좋은 따뜻한 작품입니다.,가족과 함께 보기 좋은 따뜻한 작품입니다.


### 체크포인트 2

NFC 정규화를 하지 않으면 **중복 제거**, **검색**, **모델 입력** 중 하나에서 어떤 문제가 발생할 수 있는지 예를 들어 설명하세요.

- '무늬'가 'ㅁㅜㄴㅡㅣ'로 된다면, 중복 제거나 검색, 모델 입력에서 걸리지 않습니다.

## 3. 노이즈·개인정보 패턴 탐지와 처리 정책

정규표현식은 일정한 형식의 문자열을 찾는 도구입니다. 아래 탐지는 교육용 예시이며, 실제 개인정보 탐지에는 더 엄격한 정책·검토가 필요합니다. 실습에서는 연락처를 즉시 삭제하지 않고 **마스킹**해 정보의 유형을 남깁니다.

In [9]:
def inspect_patterns(text):
    flags = []
    if re.search(r'https?://\S+|www\.\S+', text):
        flags.append('URL')
    if re.search(r'[\w.+-]+@[\w-]+\.[\w.-]+', text):
        flags.append('이메일 가능성')
    # TODO 3. 전화번호처럼 보이는 패턴을 탐지하세요.
    # 힌트: r'\d{2,3}[- ]?\d{3,4}[- ]?\d{4}'
    if re.search(r'\d{2,3}[- ]?\d{3,4}[- ]?\d{4}', text):
        flags.append('phone')

    # TODO 4. 같은 문자가 3회 이상 반복되는 패턴을 탐지하세요. 예: ㅋㅋㅋㅋ, !!!!
    # 힌트: r'(.)\1{2,}'
    if re.search(r'(.)\1{2,}', text): # (.): 임의의 문자 1개를 선택하고, 이를 그룹 '1'로 설정. \1: 그룹 '1'에서 캡처한 문자와 정확히 동일한 문자를 의미. {2,}: 앞 패턴이 최소 2번 이상 반복
        flags.append('중복 감지')

    return ', '.join(flags) if flags else '특이 패턴 없음'

if 'text_nfc' in df.columns:
    df['pattern_flags'] = df['text_nfc'].apply(inspect_patterns)
    display(df[['review_id', 'text_nfc', 'pattern_flags']])
else:
    print('TODO 2를 완료한 뒤 다시 실행하세요.')

,review_id,text_nfc,pattern_flags
0,1,정말 재미있어요!!! 배우 연기가 최고네요 😊,중복 감지
1,2,스토리는 평범하지만 음악이 좋아요.,특이 패턴 없음
2,3,기대했는데 너무너무 지루했어요...,중복 감지
3,4,영상미는 훌륭합니다. 그런데 전개가 느려요.,특이 패턴 없음
4,5,별로예요 ㅠㅠ 다시 보고 싶지 않아요.,특이 패턴 없음
5,6,문의는 movie@example.com 또는 010-1234-5678로 주세요.,"이메일 가능성, phone"
6,7,새 소식: https://example.com/movie 를 확인하세요!,URL
7,8,가족과 함께 보기 좋은 따뜻한 작품입니다.,특이 패턴 없음


In [12]:
def basic_clean(text, mask_sensitive=True): # 문맥을 파악할 때는 도움이 되면서도 정보를 공개하진 않음
    text = unicodedata.normalize('NFC', text)
    text = re.sub(r'https?://\S+|www\.\S+', ' [URL] ' if mask_sensitive else ' ', text)
    text = re.sub(r'[\w.+-]+@[\w-]+\.[\w.-]+', ' [EMAIL] ' if mask_sensitive else ' ', text)
    text = re.sub(r'\d{2,3}[- ]?\d{3,4}[- ]?\d{4}', ' [PHONE] ' if mask_sensitive else ' ', text)
    # TODO 5. 연속 공백을 하나로 줄이고 앞뒤 공백을 제거하세요.
    text = re.sub(r'\s+', ' ' if mask_sensitive else ' ', text.strip())

    return text

if 'text_nfc' in df.columns:
    df['clean_text'] = df['text_nfc'].apply(basic_clean)
    display(df[['text_nfc', 'clean_text']])
else:
    print('TODO 2를 완료한 뒤 다시 실행하세요.')

,text_nfc,clean_text
0,정말 재미있어요!!! 배우 연기가 최고네요 😊,정말 재미있어요!!! 배우 연기가 최고네요 😊
1,스토리는 평범하지만 음악이 좋아요.,스토리는 평범하지만 음악이 좋아요.
2,기대했는데 너무너무 지루했어요...,기대했는데 너무너무 지루했어요...
3,영상미는 훌륭합니다. 그런데 전개가 느려요.,영상미는 훌륭합니다. 그런데 전개가 느려요.
4,별로예요 ㅠㅠ 다시 보고 싶지 않아요.,별로예요 ㅠㅠ 다시 보고 싶지 않아요.
5,문의는 movie@example.com 또는 010-1234-5678로 주세요.,문의는 [EMAIL] 또는 [PHONE] 로 주세요.
6,새 소식: https://example.com/movie 를 확인하세요!,새 소식: [URL] 를 확인하세요!
7,가족과 함께 보기 좋은 따뜻한 작품입니다.,가족과 함께 보기 좋은 따뜻한 작품입니다.


### 체크포인트 3

반복 문자 또는 이모지를 일괄 삭제할 때 생길 수 있는 **정보 손실** 사례를 한 가지 작성하세요. 전화번호·이메일을 탐지한 뒤 실제 서비스에서 추가로 고려해야 할 조치도 한 가지 적으세요.

## 4. 토큰화 비교: 공백·정규표현식·형태소 분석

공백 기준 토큰화는 간단하지만 조사·어미가 결합된 한국어의 문법 정보를 충분히 분리하지 못합니다. Okt가 준비되지 않은 경우에도 Kiwi 결과와 공백 토큰을 비교해 관찰하세요.

In [13]:
if 'clean_text' not in df.columns:
    print('TODO 5를 완료한 뒤 다시 실행하세요.')
else:
    sample = df.loc[0, 'clean_text']
    print('원문:', sample)
    print('공백 토큰:', sample.split())
    print('정규표현식 토큰:', re.findall(r'[가-힣A-Za-z0-9]+', sample))
    print('\nKiwi 형태소:', [(token.form, token.tag) for token in kiwi.tokenize(sample)]) # NNG만 보면 '배우', '연기', '최고'.
    if okt_ready:
        print('Okt 형태소:', okt.pos(sample)) # Noun만 보면 '정말', '배우', '연기', '최고'
    else:
        print('Okt 형태소: 현재 환경에서는 비교 생략')
    # 약간의 차이 있. 분석 목적에 따라 선택하면 됨.

원문: 정말 재미있어요!!! 배우 연기가 최고네요 😊
공백 토큰: ['정말', '재미있어요!!!', '배우', '연기가', '최고네요', '😊']
정규표현식 토큰: ['정말', '재미있어요', '배우', '연기가', '최고네요']

Kiwi 형태소: [('정말', 'MAG'), ('재미있', 'VA'), ('어요', 'EF'), ('!!!', 'SF'), ('배우', 'NNG'), ('연기', 'NNG'), ('가', 'JKS'), ('최고', 'NNG'), ('이', 'VCP'), ('네요', 'EF'), ('😊', 'W_EMOJI')]
Okt 형태소: [('정말', 'Noun'), ('재미있어요', 'Adjective'), ('!!!', 'Punctuation'), ('배우', 'Noun'), ('연기', 'Noun'), ('가', 'Josa'), ('최고', 'Noun'), ('네', 'Suffix'), ('요', 'Josa'), ('😊', 'Foreign')]


In [15]:
# TODO 6. 아래 문장을 관심 있는 주제로 바꾸고, Kiwi와 (가능하면) Okt 결과를 비교하세요.
my_sentence = '최근에 백엔드 취업공고를 살펴보았다. 자바 언어만 많이 요구되어 아쉬웠다.'
print('문장:', my_sentence)
print('Kiwi:', [(t.form, t.tag) for t in kiwi.tokenize(my_sentence)]) # NNG만 보면 '최근', '백', '엔드', '취업', '공고', '언어', '요구'
if okt_ready:
    print('Okt :', okt.pos(my_sentence))
    pass
else:
    print('Okt : 현재 환경에서는 비교 생략')

문장: 최근에 백엔드 취업공고를 살펴보았다. 자바 언어만 많이 요구되어 아쉬웠다.
Kiwi: [('최근', 'NNG'), ('에', 'JKB'), ('백', 'NNG'), ('엔드', 'NNG'), ('취업', 'NNG'), ('공고', 'NNG'), ('를', 'JKO'), ('살펴보', 'VV'), ('었', 'EP'), ('다', 'EF'), ('.', 'SF'), ('자바', 'NNP'), ('언어', 'NNG'), ('만', 'JX'), ('많이', 'MAG'), ('요구', 'NNG'), ('되', 'XSV'), ('어', 'EC'), ('아쉽', 'VA-I'), ('었', 'EP'), ('다', 'EF'), ('.', 'SF')]
Okt : [('최근', 'Noun'), ('에', 'Josa'), ('백엔드', 'Noun'), ('취업', 'Noun'), ('공고', 'Noun'), ('를', 'Josa'), ('살펴보았다', 'Verb'), ('.', 'Punctuation'), ('자바', 'Noun'), ('언어', 'Noun'), ('만', 'Josa'), ('많이', 'Adverb'), ('요구', 'Noun'), ('되어', 'Verb'), ('아쉬웠다', 'Adjective'), ('.', 'Punctuation')]


### 체크포인트 4

본인 문장에서 Kiwi와 Okt가 다르게 나눈 토큰 또는 다르게 태깅한 사례를 하나 기록하세요. Okt를 사용할 수 없었다면 Kiwi 결과에서 조사·어미가 어떻게 분리되었는지 기록하세요. 결과의 우열이 아니라, **어떤 과업에 더 편리할지**를 함께 설명하세요.

## 5. 품사 선택과 불용어 처리

키워드 추출에는 명사 중심 토큰이 유용할 수 있습니다. 반면 감성 분석에서는 부정 표현·부사·어미가 중요할 수 있습니다. 아래에서는 먼저 명사를 추출한 뒤, **도메인 불용어**를 적용합니다. 조사는 이미 품사 필터에서 제거되므로, 명사 목록에 조사 불용어를 넣어도 효과가 거의 없다는 점을 확인하세요.

In [16]:
def kiwi_nouns(text):
    # TODO 7. Kiwi 토큰 중 품사 태그가 N으로 시작하는 토큰의 form만 반환하세요.

    return [token.form for token in kiwi.tokenize(text) if token.tag.startswith('N')]

if 'clean_text' in df.columns:
    df['nouns'] = df['clean_text'].apply(kiwi_nouns)
    display(df[['review_id', 'clean_text', 'nouns']])
else:
    print('TODO 5를 완료한 뒤 다시 실행하세요.')

,review_id,clean_text,nouns
0,1,정말 재미있어요!!! 배우 연기가 최고네요 😊,"[배우, 연기, 최고]"
1,2,스토리는 평범하지만 음악이 좋아요.,"[스토리, 음악]"
2,3,기대했는데 너무너무 지루했어요...,[기대]
3,4,영상미는 훌륭합니다. 그런데 전개가 느려요.,"[영상미, 전개]"
4,5,별로예요 ㅠㅠ 다시 보고 싶지 않아요.,[]
5,6,문의는 [EMAIL] 또는 [PHONE] 로 주세요.,[문의]
6,7,새 소식: [URL] 를 확인하세요!,"[소식, 확인]"
7,8,가족과 함께 보기 좋은 따뜻한 작품입니다.,"[가족, 작품]"


In [18]:
# 과업에 따라 수정할 수 있는 도메인 불용어 예시입니다.
stopwords = {'영화', '작품'}

# TODO 8. nouns에서 stopwords에 없는 토큰만 남긴 filtered_nouns 열을 만드세요.
df['filtered_nouns'] = df['nouns'].apply(lambda tokens: [token for token in tokens if token not in stopwords]) # 각 행에 대해, stopwords에 없는 애들만 뽑아 새로운 열 만듦

if 'filtered_nouns' in df.columns:
    display(df[['clean_text', 'nouns', 'filtered_nouns']]) # 7번에서 '작품' 빠짐
else:
    print('TODO 8을 완료한 뒤 다시 실행하세요.')

,clean_text,nouns,filtered_nouns
0,정말 재미있어요!!! 배우 연기가 최고네요 😊,"[배우, 연기, 최고]","[배우, 연기, 최고]"
1,스토리는 평범하지만 음악이 좋아요.,"[스토리, 음악]","[스토리, 음악]"
2,기대했는데 너무너무 지루했어요...,[기대],[기대]
3,영상미는 훌륭합니다. 그런데 전개가 느려요.,"[영상미, 전개]","[영상미, 전개]"
4,별로예요 ㅠㅠ 다시 보고 싶지 않아요.,[],[]
5,문의는 [EMAIL] 또는 [PHONE] 로 주세요.,[문의],[문의]
6,새 소식: [URL] 를 확인하세요!,"[소식, 확인]","[소식, 확인]"
7,가족과 함께 보기 좋은 따뜻한 작품입니다.,"[가족, 작품]",[가족]


In [19]:
# TODO 9. filtered_nouns의 모든 토큰을 모아 빈도를 계산하고 상위 10개를 출력하세요.
all_nouns = [
    token
    for tokens in df['filtered_nouns']
    for token in tokens
]
noun_freq = Counter(all_nouns)
display(pd.DataFrame(noun_freq.most_common(10), columns=['token', 'count']))

# 빈도로는 알 수 없는 것들이 있다. 실제로는 부정이 많은데, 아래 표를 보면 선호가 괜찮아보인다.

,token,count
0,배우,1
1,연기,1
2,최고,1
3,스토리,1
4,음악,1
5,기대,1
6,영상미,1
7,전개,1
8,문의,1
9,소식,1


### 체크포인트 5

`좋지 않아요`, `너무 지루했어요`와 같은 감성 표현을 생각해 보세요. 명사만 남기면 어떤 정보가 사라질 수 있나요? 감성 분석과 키워드 추출 중 어느 과업에 명사 중심 처리가 더 적합한지 이유와 함께 답하세요.

- 5번의 경우 추출된 게 없다. 감성 분석에서는 5번이 필요할 것이다... 사람이 들어가서 이건 넣고 저건 빼고 해야 하는 부분이 있다.
- 명사만 남기면, 정보가 사라질 수 있다. 목적을 잘 생각해서, 감정분석이면 다른 것도 남겨야겠다.

## 6. 전처리 전·후 비교

전처리의 효과는 단순히 “짧아졌다”가 아니라, 과업에 필요한 정보가 적절히 남았는지로 판단해야 합니다(사라졌으면 다시 위로 가서 살려야 합니다).

In [20]:
if 'filtered_nouns' in df.columns:
    comparison = df[['review_id', 'text', 'clean_text', 'filtered_nouns']].copy()
    comparison['original_length'] = comparison['text'].str.len()
    comparison['clean_length'] = comparison['clean_text'].str.len()
    display(comparison)
else:
    print('TODO 8을 완료한 뒤 다시 실행하세요.')

# TODO 10. 아래 질문에 Markdown 셀로 답하세요.
# 1) 한 리뷰를 골라 전처리 전·후 달라진 부분을 설명하세요.
# 2) 해당 변화가 목적(키워드 추출 또는 감성 분석)에 적합한지 판단하세요.
# 3) 삭제하지 말아야 했을 수 있는 정보가 있다면 적으세요.


# 3번은 감성분석이구나. 근데 지루했어요 같은 게 빠져서 감정분석에 사용할 수 없다.

,review_id,text,clean_text,filtered_nouns,original_length,clean_length
0,1,정말 재미있어요!!! 배우 연기가 최고네요 😊,정말 재미있어요!!! 배우 연기가 최고네요 😊,"[배우, 연기, 최고]",25,25
1,2,스토리는 평범하지만 음악이 좋아요.,스토리는 평범하지만 음악이 좋아요.,"[스토리, 음악]",19,19
2,3,기대했는데 너무너무 지루했어요...,기대했는데 너무너무 지루했어요...,[기대],19,19
3,4,영상미는 훌륭합니다. 그런데 전개가 느려요.,영상미는 훌륭합니다. 그런데 전개가 느려요.,"[영상미, 전개]",24,24
4,5,별로예요 ㅠㅠ 다시 보고 싶지 않아요.,별로예요 ㅠㅠ 다시 보고 싶지 않아요.,[],21,21
5,6,문의는 movie@example.com 또는 010-1234-5678로 주세요.,문의는 [EMAIL] 또는 [PHONE] 로 주세요.,[문의],44,29
6,7,새 소식: https://example.com/movie 를 확인하세요!,새 소식: [URL] 를 확인하세요!,"[소식, 확인]",40,20
7,8,가족과 함께 보기 좋은 따뜻한 작품입니다.,가족과 함께 보기 좋은 따뜻한 작품입니다.,[가족],23,23


- 1) 2번 리뷰는 전처리 전후에 filtered_nouns로 [스토리, 음악]이 뽑혔습니다.
- 2) 영화 관람 시 주의깊게 봐야 할 포인트라는 점에서 키워드 추출에 적합해보입니다.
- 3) '평범'과 '좋아요'가 있다면 더 좋을 것 같습니다.
- 중간평가는 감성분석. 데이터를 찾아서 해야하는데
나는 뭘 해야 할까. 품사 뭘 남겨서 어떤 식으로 해야 유리할까를 생각해보자.

## 제출 체크리스트

- [ ] TODO 1~10을 작성하고 실행했다.
- [ ] NFC 정규화의 필요성을 설명했다.
- [ ] URL·이메일·전화번호·반복 문자 중 2개 이상을 탐지했다.
- [ ] 민감정보를 마스킹할 때와 삭제할 때의 차이를 설명했다.
- [ ] 공백·정규표현식·Kiwi·(가능하면) Okt 토큰화 결과를 비교했다.
- [ ] 명사 중심 처리와 도메인 불용어 처리의 한계를 설명했다.
- [ ] 전처리 전·후 비교 결과와 판단 근거를 기록했다.
- [ ] 외부 AI/자료 사용 시 사용 범위와 검증 과정을 기록했다.

### 심화 과제(선택)
1. `basic_clean`에서 이모지·반복 문자 수를 별도 특징으로 보존하세요.
2. 감성 표현을 보존하려면 명사 외에 어떤 품사를 남겨야 하는지 규칙을 제안·구현하세요.
3. 직접 수집한 텍스트 5건 이상에 같은 규칙을 적용하고, 출처·라이선스·개인정보 처리 방침을 기록하세요.

## 다음 주차 예고

다음 주차에는 전처리된 텍스트를 컴퓨터가 계산할 수 있는 수치 벡터로 표현합니다. 단어 빈도, Bag-of-Words, TF-IDF를 이용해 문서를 비교하고 분류 모델의 입력 특징을 구성합니다.